# Does the toy's dlog-Fourier algorithm transfer to Llama?

Part 4 of the *Manifold Features* series. Tests whether the toy's modular-division algorithm from Part 1 (clean dlog-Fourier circles with joint R² > 0.93 at P=97) is present in Llama-3.1-8B.

For each prime `P ∈ {3, 5, 7, 11, 13, 23, 97}` we build every triple `(a, b, c)` with `c = a · b⁻¹ mod P` via Fermat's little theorem, prompt Llama, capture the L=18 last-token residual, then fit Fourier probes in two bases:

- **Natural basis** — sin/cos of `c` mod P (a c-cycle).
- **Dlog basis** — sin/cos of `dlog_g(c)` mod (P-1), with g the smallest primitive root. This is the basis the toy uses.

Sanity probes for the inputs `a` and `b` (Llama must at least see the visible numbers).

**Source model**: `meta-llama/Llama-3.1-8B` (gated; needs `HF_TOKEN`).

**Outputs**: `./out/toy_algorithms_absence/results.json`, `./out/toy_algorithms_absence/summary.png`, `./out/toy_algorithms_absence/centroids_P{P}.png`.

**Runs on**: a single Colab L4 (24 GB) in ~10 minutes. Set `RECOMPUTE = False` to skip the model load and replot from the bundled `../data/llama_modular_division_results.json` in a few seconds.


In [ ]:
# Cell 1 — setup
import os, sys, json, time, subprocess
from pathlib import Path

def _pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

try:
    import numpy as np
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from torch.nn.functional import softmax
    from sklearn.linear_model import Ridge
    import matplotlib.pyplot as plt
except ImportError:
    _pip('torch', 'transformers', 'numpy', 'scikit-learn', 'matplotlib', 'huggingface_hub')
    import numpy as np
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from torch.nn.functional import softmax
    from sklearn.linear_model import Ridge
    import matplotlib.pyplot as plt

# HF token resolution (Colab userdata, then env)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
if HF_TOKEN is None:
    raise RuntimeError('Llama 3.1 8B is gated. Set HF_TOKEN as an env var or Colab secret.')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'meta-llama/Llama-3.1-8B'
MAX_NEW_TOKENS = 512

BG, FG, GRID, MUTED = '#FAFAF7', '#2A2A2A', '#E5E3DD', '#6E6E6E'
plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG, 'savefig.facecolor': BG,
    'axes.edgecolor': FG, 'axes.linewidth': 0.5, 'axes.labelcolor': FG,
    'axes.titlecolor': FG, 'xtick.color': FG, 'ytick.color': FG, 'text.color': FG,
    'font.family': 'sans-serif', 'font.sans-serif': ['Helvetica Neue', 'Helvetica', 'DejaVu Sans'],
    'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.5, 'grid.alpha': 0.6,
})


RECOMPUTE = True   # set False to replot from bundled JSON without loading Llama

OUT_DIR = Path('./out/toy_algorithms_absence').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_LAYER = 18
RIDGE_ALPHA = 100.0
SEED = 42
MODULI = [3, 5, 7, 11, 13, 23, 97]

DATA_FALLBACK = Path('../data/llama_modular_division_results.json')


In [ ]:
# Cell 2 — modular-division dataset + primitive-root / dlog tables per P
def build_div_pairs(P, max_pairs=400):
    # All (a, b, c) with c = a * b^(-1) mod P; enumerated for small P, sampled otherwise.
    pairs = []
    if P * P <= max_pairs * 2:
        for a in range(1, P):
            for b in range(1, P):
                b_inv = pow(b, P - 2, P)
                pairs.append((a, b, (a * b_inv) % P))
    else:
        rng = np.random.default_rng(SEED)
        seen = set()
        while len(pairs) < max_pairs:
            a, b = int(rng.integers(1, P)), int(rng.integers(1, P))
            if (a, b) in seen:
                continue
            seen.add((a, b))
            pairs.append((a, b, (a * pow(b, P - 2, P)) % P))
    return pairs


def find_generator(P):
    # Smallest primitive root g of Z/P*, for prime P.
    pm1 = P - 1
    n, factors = pm1, []
    f = 2
    while f * f <= n:
        if n % f == 0:
            factors.append(f)
            while n % f == 0:
                n //= f
        f += 1
    if n > 1:
        factors.append(n)
    for g in range(2, P):
        if all(pow(g, pm1 // q, P) != 1 for q in factors):
            return g
    return None


def build_dlog_table(P, g):
    arr = np.zeros(P, dtype=np.int64)
    arr[0] = -1   # undefined
    x = 1
    for i in range(P - 1):
        arr[x] = i
        x = (x * g) % P
    return arr


datasets = {}
for P in MODULI:
    pairs = build_div_pairs(P)
    g = find_generator(P)
    dlog = build_dlog_table(P, g) if g else None
    datasets[P] = {'pairs': pairs, 'g': g, 'dlog': dlog}
    print(f'P={P:>3d}: {len(pairs)} pairs, g={g}, '
          f'dlog(1..{min(P-1, 6)})={list(dlog[1:min(P, 7)]) if dlog is not None else None}')


In [ ]:
# Cell 3 — load Llama (only if RECOMPUTE) + helpers
if RECOMPUTE:
    # Load Llama-3.1-8B (gated; needs HF_TOKEN)
    tok = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tok.padding_side = 'left'
    if tok.pad_token_id is None:
        tok.pad_token_id = tok.eos_token_id
    
    t0 = time.time()
    MODEL = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.bfloat16,
        attn_implementation='sdpa', device_map='auto', token=HF_TOKEN,
    ).eval()
    for p in MODEL.parameters():
        p.requires_grad_(False)
    print(f'Loaded {MODEL_NAME} in {time.time()-t0:.1f}s on {DEVICE}')
    


    @torch.no_grad()
    def batch_argmax(prompts, batch_size=16):
        out_tokens = []
        for s in range(0, len(prompts), batch_size):
            e = min(s + batch_size, len(prompts))
            ids = tok(prompts[s:e], return_tensors='pt', padding=True).to(DEVICE)
            logits = MODEL(**ids).logits
            out_tokens.extend(logits[:, -1].argmax(-1).cpu().tolist())
        return out_tokens

    def parse_int(tid):
        s = tok.decode(tid).strip()
        try: return int(s)
        except ValueError: return None

    @torch.no_grad()
    def capture_layer(prompts, layer, batch_size=16):
        acts = []
        for s in range(0, len(prompts), batch_size):
            e = min(s + batch_size, len(prompts))
            ids = tok(prompts[s:e], return_tensors='pt', padding=True).to(DEVICE)
            captured = [None]
            def hk(module, inp, out):
                h = out[0] if isinstance(out, tuple) else out
                captured[0] = h[:, -1, :].detach().float().cpu().numpy()
            handle = MODEL.model.layers[layer].register_forward_hook(hk)
            try: MODEL(**ids)
            finally: handle.remove()
            acts.append(captured[0])
        return np.concatenate(acts, axis=0)

    print('Helpers ready')
else:
    print('RECOMPUTE = False; will replot from bundled JSON in the last cell.')


In [ ]:
# Cell 4 — find best prompt template per P, measure baseline accuracy
PROMPT_TEMPLATES = [
    lambda a, b, P: f'({a} / {b}) mod {P} = ',
    lambda a, b, P: f'What is the modular inverse of {b} modulo {P}, times {a}? Answer: ',
    lambda a, b, P: f'Find x such that {b} * x ≡ {a} (mod {P}). x = ',
    lambda a, b, P: f'{a} * {b}^(-1) mod {P} = ',
]

best_prompts = {}
if RECOMPUTE:
    print('Probing baseline accuracy per (P, template):')
    for P in MODULI:
        pairs = datasets[P]['pairs']
        rng = np.random.default_rng(0)
        sub = [pairs[i] for i in rng.choice(len(pairs), size=min(30, len(pairs)), replace=False)]
        scores = []
        for ti, tmpl in enumerate(PROMPT_TEMPLATES):
            prompts = [tmpl(a, b, P) for a, b, c in sub]
            tids = batch_argmax(prompts)
            n_ok = sum(1 for tid, (a, b, c) in zip(tids, sub) if parse_int(tid) == c)
            n_trivial = sum(1 for tid, (a, b, c) in zip(tids, sub)
                            if parse_int(tid) in (a, b, a + b))
            scores.append((ti, n_ok / len(sub), n_trivial / len(sub)))
        best_ti, best_acc, best_trivial = max(scores, key=lambda x: x[1])
        best_prompts[P] = {'template_idx': best_ti, 'baseline_acc': best_acc,
                           'trivial_rate': best_trivial}
        print(f'  P={P:>3d}: best template {best_ti}, baseline c-correct={best_acc:.2f} '
              f'(chance = 1/{P-1} = {1/(P-1):.3f})')


In [ ]:
# Cell 5 — capture L=18 residuals for all pairs
captures = {}
if RECOMPUTE:
    for P in MODULI:
        pairs = datasets[P]['pairs']
        ti = best_prompts[P]['template_idx']
        prompts = [PROMPT_TEMPLATES[ti](a, b, P) for a, b, c in pairs]
        t0 = time.time()
        R = capture_layer(prompts, TARGET_LAYER)
        captures[P] = {
            'R': R,
            'a': np.array([p[0] for p in pairs]),
            'b': np.array([p[1] for p in pairs]),
            'c': np.array([p[2] for p in pairs]),
        }
        print(f'P={P:>3d}: residuals shape {R.shape} in {time.time()-t0:.0f}s')


In [ ]:
# Cell 6 — train Fourier probes for c in two bases (natural + dlog)
def train_probe_pair(R, targets, T, alpha=RIDGE_ALPHA, test_frac=0.2, seed=SEED):
    n = len(targets)
    if n < 10: return None
    targets = np.asarray(targets, dtype=np.float64)
    rng = np.random.RandomState(seed)
    idx = rng.permutation(n)
    n_tr = int(n * (1 - test_frac))
    tr, te = idx[:n_tr], idx[n_tr:]
    sin_t = np.sin(2 * np.pi * targets / T)
    cos_t = np.cos(2 * np.pi * targets / T)
    if sin_t[tr].std() < 1e-6 or cos_t[tr].std() < 1e-6:
        return None
    sp = Ridge(alpha=alpha).fit(R[tr], sin_t[tr])
    cp = Ridge(alpha=alpha).fit(R[tr], cos_t[tr])
    sin_pred = sp.predict(R[te]); cos_pred = cp.predict(R[te])
    joint_var = sin_t[te].var() + cos_t[te].var()
    if joint_var < 1e-6: return None
    joint_res = ((sin_pred - sin_t[te]) ** 2).mean() + ((cos_pred - cos_t[te]) ** 2).mean()
    return float(1 - joint_res / joint_var)


import math
def clean(v):
    if v is None: return None
    if isinstance(v, float) and (math.isinf(v) or math.isnan(v)): return None
    return v


results = {}
if RECOMPUTE:
    print(f'\n{"P":>3s}  {"baseline":>9s}  {"R²(c nat)":>11s}  {"R²(dlog c)":>11s}  {"R²(a)":>7s}  {"R²(b)":>7s}')
    print('-' * 65)
    for P in MODULI:
        d = captures[P]; R = d['R']
        a, b, c = d['a'], d['b'], d['c']
        dlog = datasets[P]['dlog']
        r2_c_nat = train_probe_pair(R, c, T=P)
        r2_c_dlog = train_probe_pair(R, dlog[c], T=P - 1) if dlog is not None else None
        r2_a = train_probe_pair(R, a, T=P)
        r2_b = train_probe_pair(R, b, T=P)
        results[P] = {
            'baseline_acc': best_prompts[P]['baseline_acc'],
            'trivial_rate': best_prompts[P]['trivial_rate'],
            'r2_c_natural': r2_c_nat,
            'r2_c_dlog': r2_c_dlog,
            'r2_a': r2_a, 'r2_b': r2_b,
        }
        fmt = lambda v: f'{v:+.3f}' if v is not None else '  N/A '
        print(f'{P:>3d}  {best_prompts[P]["baseline_acc"]:>9.2f}  {fmt(r2_c_nat):>11s}  '
              f'{fmt(r2_c_dlog):>11s}  {fmt(r2_a):>7s}  {fmt(r2_b):>7s}')

    with open(OUT_DIR / 'results.json', 'w') as f:
        json.dump({str(P): {k: clean(v) for k, v in r.items()} for P, r in results.items()},
                  f, indent=2)
    print(f'\nSaved {OUT_DIR/"results.json"}')
else:
    with open(DATA_FALLBACK) as f:
        raw = json.load(f)
    results = {int(P): r for P, r in raw.items()}
    print(f'Loaded {len(results)} P results from {DATA_FALLBACK}')


In [ ]:
# Cell 7 — Figure: baseline accuracy + Fourier probe quality (the two-panel summary)
Ps = sorted(results.keys())
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
x = np.arange(len(Ps)); w = 0.30

# Left: probe R² in both bases + input sanity
r2_nat = [results[P]['r2_c_natural'] or 0 for P in Ps]
r2_dlog = [results[P]['r2_c_dlog'] or 0 for P in Ps]
r2_a = [results[P]['r2_a'] or 0 for P in Ps]
r2_b = [results[P]['r2_b'] or 0 for P in Ps]

axes[0].bar(x - 1.5*w, r2_nat,  w, label='R²(c natural)',         color='#cc4444', edgecolor='black')
axes[0].bar(x - 0.5*w, r2_dlog, w, label='R²(dlog c) [toy-style]', color='#aa44aa', edgecolor='black')
axes[0].bar(x + 0.5*w, r2_a,    w, label='R²(a) [input sanity]',   color='#888888', edgecolor='black', alpha=0.7)
axes[0].bar(x + 1.5*w, r2_b,    w, label='R²(b) [input sanity]',   color='#bbbbbb', edgecolor='black', alpha=0.7)
axes[0].axhline(0.5, ls='--', color='gray', alpha=0.5, label='strong-probe threshold')
axes[0].axhline(0.93, ls=':', color='#aa44aa', alpha=0.6, label='toy R²(dlog c) at P=97')
axes[0].set_xticks(x); axes[0].set_xticklabels([f'P={p}' for p in Ps])
axes[0].set_ylabel('Probe joint R²')
axes[0].set_title('Toy-style dlog-Fourier circles at L=18?\nLlama vs the toy reference (Part 1)')
axes[0].legend(fontsize=8, loc='lower left')
axes[0].set_ylim(-0.3, 1.05)

# Right: baseline accuracy vs chance
baselines = [results[P]['baseline_acc'] for P in Ps]
chance = [1.0 / (P - 1) for P in Ps]
axes[1].bar(x - 0.2, baselines, 0.4, label='Llama baseline',     color='#888888', edgecolor='black')
axes[1].bar(x + 0.2, chance,    0.4, label='chance = 1/(P-1)',   color='lightcoral', edgecolor='black', alpha=0.6)
axes[1].set_xticks(x); axes[1].set_xticklabels([f'P={p}' for p in Ps])
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Llama baseline on (a / b) mod P\n(modular division: no trivial fallback)')
axes[1].set_ylim(0, 1.05); axes[1].legend()
for i, v in enumerate(baselines): axes[1].annotate(f'{v:.2f}', (i - 0.2, v + 0.02), ha='center', fontsize=8)
for i, v in enumerate(chance):    axes[1].annotate(f'{v:.2f}', (i + 0.2, v + 0.02), ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / 'summary.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved {OUT_DIR/"summary.png"}')


## Expected results

- **R²(c natural)** and **R²(dlog c)** stay below 0.3 for every prime — neither basis recovers the modular-division answer from the L=18 residual.
- **R²(a)** and **R²(b)** are positive (the model sees the inputs).
- **Baseline accuracy** drops from 100% at P=3 (one nontrivial divisor) to ~3% at P=97, tracking 1/(P-1) chance closely.

Compare against the toy from Part 1, where the dlog-basis probe hit joint R² > 0.93 at P=97 and the model grokked the task. The toy's specific algorithm is not in Llama — what transfers from toy to LLM is the *kind* of object (multi-frequency Fourier counting), not the specific dlog-mod-97 circles. The next notebook (`goodfire_replication.ipynb`) extracts the basis that Llama actually does carry: six base-10 clocks at the same L=18 layer.
